# Run the pi harness with the OpenAI API

This notebook runs the **pi fork's source code** in a workspace of your choice.
Configure the model, task, skills, and additional system instructions, then inspect
the response, usage, trajectory, and deliverables.

Open it in VS Code or Jupyter with a **Python 3.10+** kernel and run the cells in order.
The "Run pi" cell makes real OpenAI API calls. The committed notebook contains no outputs or saved API keys.

Prepare the checkout once, using Node.js 22.19+:

```bash
cd /path/to/pi
npm ci --ignore-scripts
npm run hydrate:model-data
```

The wrapper uses only the Python standard library.
[Harness documentation](../packages/coding-agent/docs/python-harness.md).


## 1. Load the fork's wrapper

The checkout is discovered in the current directory or its parents. If your kernel
starts elsewhere, replace `PI_REPO = None` with the path to your checkout.


In [ ]:
import json
import os
import sys
from getpass import getpass
from pathlib import Path
from tempfile import mkdtemp

PI_REPO = None  # Example: Path("/path/to/pi")

if PI_REPO is None:
    PI_REPO = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "scripts/pi_harness.py").is_file()),
        None,
    )
if PI_REPO is None:
    raise FileNotFoundError("Set PI_REPO to the path of your pi fork.")

PI_REPO = Path(PI_REPO).expanduser().resolve()
if not (PI_REPO / "scripts/pi_harness.py").is_file():
    raise FileNotFoundError(f"Wrapper not found in {PI_REPO}")
sys.path.insert(0, str(PI_REPO / "scripts"))

from pi_harness import PiRunError, PiRunTimeout, run_pi

print(f"Harness loaded from: {PI_REPO}")


## 2. Configure OpenAI access

If `OPENAI_API_KEY` is already set in the kernel's environment, it is preserved.
Otherwise, the masked input below configures it for this Python session.
Do not paste the key into a code cell.


In [ ]:
if not os.environ.get("OPENAI_API_KEY", "").strip():
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ").strip()
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("An OpenAI API key is required.")

print("API key configured for this kernel; its value is not displayed.")


## 3. Choose the model, workspace, and task

- `WORKSPACE = None` creates a small example with instructions, a CSV file, and a local skill.
- To use your own files, set an **existing workspace** path and update `TASK_PROMPT`.
- `SKILL_PATHS` adds skills to those pi already discovers in the workspace and personal configuration.
- `APPEND_SYSTEM_PROMPT` adds instructions to the existing system-prompt complements.
- `TIMEOUT_SECONDS` covers startup and the task; `None` imposes no deadline.

The model must be available to your API key and present in pi's catalog.


In [ ]:
MODEL = "gpt-6-sol"
WORKSPACE = None  # Example: Path("/path/to/jobbench-workspace")
SKILL_PATHS = []  # Example: ["/path/to/skills/review"]
APPEND_SYSTEM_PROMPT = "Write the deliverables in English."
TIMEOUT_SECONDS = 300

TASK_PROMPT = (
    "Read task_folder/TASK_INSTRUCTIONS.txt and the reference files. "
    "Complete the task and save the final deliverables under output/."
)


## 4. Prepare the workspace

For the demo, this cell creates a new temporary directory and keeps its files on disk.
If you supplied your own workspace, it only checks that the directory exists.


In [ ]:
if WORKSPACE is None:
    WORKSPACE = Path(mkdtemp(prefix="pi-notebook-"))
    task_folder = WORKSPACE / "task_folder"
    task_folder.mkdir()
    (WORKSPACE / "output").mkdir()
    (WORKSPACE / "AGENTS.md").write_text(
        "Keep the reference files unchanged. Write final deliverables under output/.\n",
        encoding="utf-8",
    )
    (task_folder / "TASK_INSTRUCTIONS.txt").write_text(
        "Analyze task_folder/sales.csv. Calculate revenue per product and the total. "
        "Write output/report.md and explain your calculations.\n",
        encoding="utf-8",
    )
    (task_folder / "sales.csv").write_text(
        "product,units,unit_price\nA,4,12.50\nB,3,20.00\n", encoding="utf-8"
    )
    skill_dir = WORKSPACE / ".pi/skills/revenue-analysis"
    skill_dir.mkdir(parents=True)
    (skill_dir / "SKILL.md").write_text(
        "---\nname: revenue-analysis\n"
        "description: Calculate revenue from sales CSV files.\n"
        "---\nUse Python's csv module. Revenue is units * unit_price. "
        "Show per-product revenue and the grand total.\n",
        encoding="utf-8",
    )
    (WORKSPACE / ".pi/APPEND_SYSTEM.md").write_text(
        "Use the revenue-analysis skill for sales analysis tasks.\n", encoding="utf-8"
    )

WORKSPACE = Path(WORKSPACE).expanduser().resolve(strict=True)
if not WORKSPACE.is_dir():
    raise NotADirectoryError(WORKSPACE)

print(f"Agent workspace: {WORKSPACE}")
print(f"Model: {MODEL}")
print(f"Task: {TASK_PROMPT}")


## 5. Run pi

**This cell makes API calls.** Python waits for the full agent loop, including tool calls.
The harness is loaded from `PI_REPO`, while Node and the tools work in `WORKSPACE`.

The timeout stops the run and its child processes. Interrupting the kernel with
the stop button also triggers the wrapper's cleanup.


In [ ]:
result = None
try:
    result = run_pi(
        prompt=TASK_PROMPT,
        cwd=WORKSPACE,
        model=MODEL,
        skill_paths=SKILL_PATHS,
        append_system_prompt=APPEND_SYSTEM_PROMPT,
        timeout=TIMEOUT_SECONDS,
    )
except PiRunError as error:  # Includes PiRunTimeout.
    print(f"Run failed: {error}")
    print(error.stderr)
    raise

print(result["response"])


## 6. Inspect usage and trajectory

These are pi's native statistics: tokens, estimated cost, tool calls, and session details.
The trajectory contains native messages, tool calls, and tool results.


In [ ]:
if result is None:
    raise RuntimeError("Complete the run cell successfully first.")
print(json.dumps(result["stats"], ensure_ascii=False, indent=2))


In [ ]:
if result is None:
    raise RuntimeError("Complete the run cell successfully first.")
result["messages"]


## 7. Inspect deliverables

Files remain in the workspace, including after a failure or timeout. This cell lists
the files produced under `output/` and displays the demo's Markdown report if it exists.


In [ ]:
output_dir = WORKSPACE / "output"
deliverables = sorted(path for path in output_dir.rglob("*") if path.is_file())
print(f"Output directory: {output_dir}")
for path in deliverables:
    print(path.relative_to(WORKSPACE))

report_path = output_dir / "report.md"
if report_path.is_file():
    print("\n" + report_path.read_text(encoding="utf-8"))


## Run another candidate

Each `run_pi` call creates a fresh session and reloads the fork's sources without
a rebuild. You can modify the harness and then run the launch cell again.
Workspace files are preserved between runs: prepare a new copy of the task if
you want to start from the same initial data.

To create another demo workspace, set `WORKSPACE = None` and rerun the preparation cell.
This notebook does not download JobBench or run its judge.
